In [ ]:
!python src/extrapolate/gnn_extrapolate_copy.py # original

## No label info

In [ ]:
!python src/extrapolate/gnn_extrapolate_copy.py

## Only label no embedding

In [ ]:
!python src/extrapolate/gnn_extrapolate_copy.py

## GCN2 Conv 3 layers

In [ ]:
!python src/extrapolate/gnn_extrapolate_copy.py # this is pure GCN

In [ ]:
!python src/extrapolate/gnn_extrapolate_copy.py # cached

In [ ]:
import torch
import json
import numpy as np
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Load embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

embeddings = torch.load("/nfs/homedirs/dhp/unsupervised-data-pruning/savedir/embeddings/CIFAR10/embeddings_dict.pth", map_location=device)  # Update path

# Convert to numpy
embeddings_np = embeddings.cpu().numpy()  # Shape: (1823843, 2048)

# Load scores
with open("/nfs/homedirs/dhp/unsupervised-data-pruning/scores/prune/CIFAR10_dynamic_uncertainty_0.json") as f:  # Update path
    full_scores_dict = json.load(f)

# Ensure the scores align with embeddings
scores = np.array([full_scores_dict[str(i)] for i in range(len(embeddings_np))])  # Assuming scores are indexed

# Reduce dimensions with PCA (to 50 for speed) then apply t-SNE
pca = PCA(n_components=50)
embeddings_pca = pca.fit_transform(embeddings_np)

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_pca)

# Normalize scores for color mapping
scores_normalized = (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

# Create interactive plot with Plotly
fig = px.scatter(
    x=embeddings_2d[:, 0], 
    y=embeddings_2d[:, 1], 
    color=scores_normalized,
    title="t-SNE Places 365 DU Original",
    labels={"x": "t-SNE Component 1", "y": "t-SNE Component 2", "color": "Score"},
    color_continuous_scale="viridis"
)

fig.show()

# save fig as html in /nfs/homedirs/dhp/unsupervised-data-pruning/reports
fig.write_html("/nfs/homedirs/dhp/unsupervised-data-pruning/reports/tsne_visualization_original_CIFAR.html")



In [ ]:
import torch
import json
import numpy as np
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Load embeddings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

embeddings = torch.load("/nfs/homedirs/dhp/unsupervised-data-pruning/savedir/embeddings/CIFAR10/embeddings_dict.pth", map_location=device)  # Update path

# Convert to numpy
embeddings_np = embeddings.cpu().numpy()  # Shape: (1823843, 2048)

# Load scores
with open("/nfs/homedirs/dhp/unsupervised-data-pruning/scores/extrapolation/extrapolated/gnn_extrapolation_CIFAR10_resnet50-self-trained_k_20_seed_20000_euclidean.json") as f:  # Update path
    full_scores_dict = json.load(f)

# Ensure the scores align with embeddings
scores = np.array([full_scores_dict[str(i)] for i in range(len(embeddings_np))])  # Assuming scores are indexed

# Reduce dimensions with PCA (to 50 for speed) then apply t-SNE
pca = PCA(n_components=50)
embeddings_pca = pca.fit_transform(embeddings_np)

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_pca)

# Normalize scores for color mapping
scores_normalized = (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

# Create interactive plot with Plotly
fig = px.scatter(
    x=embeddings_2d[:, 0], 
    y=embeddings_2d[:, 1], 
    color=scores_normalized,
    title="Places 365 DU Extrapolated",
    labels={"x": "t-SNE Component 1", "y": "t-SNE Component 2", "color": "Score"},
    color_continuous_scale="viridis"
)

fig.show()

# save fig as html in /nfs/homedirs/dhp/unsupervised-data-pruning/reports
fig.write_html("/nfs/homedirs/dhp/unsupervised-data-pruning/reports/tsne_visualization_extrapolated_CIFAR.html")

In [ ]:
import json
import os
import sys

import numpy as np
import torch

sys.path.append(os.path.join(os.path.dirname(__file__), ".."))

from utils.dataset import prepare_data

trainset, train_loader, _, num_samples = prepare_data(cfg.dataset, 1024)
labels_tensor = torch.zeros(num_samples, dtype=torch.int64, device=device)

for _, labels, sample_idxs in train_loader:
    labels = labels.to(device)
    sample_idxs = sample_idxs.to(device)
    labels_tensor[sample_idxs] = labels

In [ ]:
import torch
import json
import numpy as np
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import os
import sys
from omegaconf import OmegaConf

sys.path.append("/nfs/homedirs/dhp/unsupervised-data-pruning/src")

# Load embeddings
cfg = OmegaConf.load("/nfs/homedirs/dhp/unsupervised-data-pruning/src/extrapolate/configs/gnn_config.yaml")
cfg = cfg.CIFAR10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

embeddings = torch.load("/nfs/homedirs/dhp/unsupervised-data-pruning/savedir/embeddings/CIFAR10/embeddings_dict.pth", map_location=device)  # Update path

# Convert to numpy
embeddings_np = embeddings.cpu().numpy()  # Shape: (1823843, 2048)

# Load scores
with open("/nfs/homedirs/dhp/unsupervised-data-pruning/scores/prune/CIFAR10_dynamic_uncertainty_0.json") as f:  # Update path
    full_scores_dict = json.load(f)

# Ensure the scores align with embeddings
scores = np.array([full_scores_dict[str(i)] for i in range(len(embeddings_np))])  # Assuming scores are indexed

# Load labels
from utils.dataset import prepare_data

trainset, train_loader, _, num_samples = prepare_data(cfg.dataset, 1024)  # Ensure this is correct
labels_tensor = torch.zeros(num_samples, dtype=torch.int64, device=device)

for _, labels, sample_idxs in train_loader:
    labels = labels.to(device)
    sample_idxs = sample_idxs.to(device)
    labels_tensor[sample_idxs] = labels

labels_np = labels_tensor.cpu().numpy()  # Convert labels to numpy

# Reduce dimensions with PCA (to 50 for speed) then apply t-SNE
pca = PCA(n_components=50)
embeddings_pca = pca.fit_transform(embeddings_np)

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_pca)

# Normalize scores for color mapping
scores_normalized = (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

# Create interactive plot with Plotly
fig = px.scatter(
    x=embeddings_2d[:, 0], 
    y=embeddings_2d[:, 1], 
    color=scores_normalized,
    title="t-SNE Projection of Scores with Labels",
    labels={"x": "t-SNE Component 1", "y": "t-SNE Component 2", "color": "Score"},
    color_continuous_scale="viridis",
    hover_data={"Label": labels_np}  # Add labels for hover info
)

fig.show()

# Save figure as HTML
fig.write_html("/nfs/homedirs/dhp/unsupervised-data-pruning/reports/tsne__original_CIFAR.html")


In [1]:
import cudf
from cuml.neighbors import NearestNeighbors
from cuml.datasets import make_blobs

X, _ = make_blobs(n_samples=5, centers=5, n_features=10, random_state=42)

X_cudf = cudf.DataFrame(X)

model = NearestNeighbors(n_neighbors=3)
model.fit(X)

distances, indices = model.kneighbors(X_cudf)

UnsupportedCUDAError: A GPU with NVIDIA Volta™ (Compute Capability 7.0) or newer architecture is required.
Detected GPU 0: NVIDIA GeForce GTX 1080 Ti                                                                                                                                                                                                                                      
Detected Compute Capability: 6.1